In [ ]:
# ── INSTALL (run once in terminal) ──────────────────────────────────────────
# pip install transformers datasets scikit-learn torch

# ────────────────────────────────────────────────────────────────────────────
# STEP 1: Explore the Dataset
# ────────────────────────────────────────────────────────────────────────────
from datasets import load_dataset

dataset = load_dataset("sms_spam", trust_remote_code=True)
print(dataset)

sample = dataset["train"][0]
print("Column names:", list(sample.keys()))

label_names = dataset["train"].features["label"].names
print("Label names:", label_names)  # ['ham', 'spam']

# ────────────────────────────────────────────────────────────────────────────
# STEP 2: Label Dictionaries
# ────────────────────────────────────────────────────────────────────────────
label_map = {0: "ham", 1: "spam"}
id_map    = {"ham": 0, "spam": 1}

# ────────────────────────────────────────────────────────────────────────────
# STEP 3: Tokenize & Preprocess
# ────────────────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_CKPT = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize(batch):
    return tokenizer(
        batch["sms"],           # sms_spam dataset uses "sms" as text column
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized = dataset.map(tokenize, batched=True)

# ────────────────────────────────────────────────────────────────────────────
# STEP 4: Split Train & Eval
# ────────────────────────────────────────────────────────────────────────────
# STEP 4: Split Train & Eval
full          = tokenized["train"].shuffle(seed=42)
train_dataset = full.select(range(4000))
eval_dataset  = full.select(range(4000, 5574))

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# ────────────────────────────────────────────────────────────────────────────
# STEP 5: Fine-Tune DistilBERT
# ────────────────────────────────────────────────────────────────────────────
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=2,
    id2label=label_map,
    label2id=id_map,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./spam_model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())

# ────────────────────────────────────────────────────────────────────────────
# STEP 6: Save Model
# ────────────────────────────────────────────────────────────────────────────
trainer.save_model("./spam_model")
tokenizer.save_pretrained("./spam_model")

# ────────────────────────────────────────────────────────────────────────────
# STEP 7: Load & Predict
# ────────────────────────────────────────────────────────────────────────────
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

loaded_tokenizer = AutoTokenizer.from_pretrained("./spam_model")
loaded_model     = AutoModelForSequenceClassification.from_pretrained("./spam_model")
loaded_model.eval()

def predict_with_label(text: str) -> str:
    inputs = loaded_tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = loaded_model(**inputs)
    pred_id = outputs.logits.argmax(dim=-1).item()
    return f"'{text}' → {label_map[pred_id].upper()}"

texts = [
    "Congratulations! You've won a free ticket.",
    "Hey, are we meeting tomorrow?",
]

for t in texts:
    print(predict_with_label(t))

C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'sms_spam' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})
Column names: ['sms', 'label']
Label names: ['ham', 'spam']
Train: 4000 | Eval: 1574
